In [1]:
import numpy as np
import xarray as xr

In [2]:

# =============================================================================
# File names
# =============================================================================

file1 = "data/gdev1420.pop.h.0281_0300.nc"  # varAll
file2 = "data/gdev1440.pop.h.0281_0300.nc"  # fixAll
file3 = "data/gdev1441.pop.h.0281_0300.nc"  # varN
file4 = "data/gdev1442.pop.h.0281_0300.nc"  # fixN
file5 = "data/gdev1443.pop.h.0281_0300.nc"  # varP
file6 = "data/gdev1444.pop.h.0281_0300.nc"  # fixP
file7 = "data/gdev1445.pop.h.0281_0300.nc"  # varFe
file8 = "data/gdev1446.pop.h.0281_0300.nc"  # fixFe
file9 = "data/gdev1447.pop.h.0281_0300.nc"  # varSi
file10 = "data/gdev1448.pop.h.0281_0300.nc"  # fixSi

files = [
    file2, file3, file5, file7, file9,
    file1, file4, file6, file8, file10
]


In [3]:

# =============================================================================
# Read grid variables
# =============================================================================

grid = xr.open_dataset(file1)

lat = grid["TLAT"]
lon = grid["TLONG"]

# Convert cm2 -> m2
tarea = grid["TAREA"] * 1e-4

# Convert cm -> m
depth = grid["z_t"] * 1e-2
depth150 = grid["z_t_150m"] * 1e-2

# Find index nearest 105 m
k100 = int(np.argmin(np.abs(depth.values - 105.0)))

# Cell volume
vol = tarea.expand_dims(z_t=depth) * depth

NPP_tot = []
POC_tot = []
PON_tot = []
POP_tot = []
Piron_tot = []
SiO2_tot = []
Nfix_tot = []
Denit_tot = []

spC_tot = []
diatC_tot = []
diazC_tot = []
biomass_tot = []


/tmp/ipykernel_11344/4094762450.py:5: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  grid = xr.open_dataset(file1)


In [4]:

# =============================================================================
# Main loop
# =============================================================================

for fname in files:

    ds = xr.open_dataset(fname)

    # -------------------------------------------------------------------------
    # NPP
    # -------------------------------------------------------------------------

    photoC_sp = ds["photoC_sp"]
    photoC_diat = ds["photoC_diat"]
    photoC_diaz = ds["photoC_diaz"]

    NPP = photoC_sp + photoC_diat + photoC_diaz

    # upper 100 m (first 11 levels)
    NPP100 = NPP.isel(z_t_150m=slice(0, 11)).sum("z_t_150m")

    # -------------------------------------------------------------------------
    # Biomass
    # -------------------------------------------------------------------------

    spC = ds["spC"]
    diatC = ds["diatC"]
    diazC = ds["diazC"]

    biomass = spC + diatC + diazC

    # -------------------------------------------------------------------------
    # Export fluxes
    # -------------------------------------------------------------------------

    POC100 = ds["POC_FLUX_IN"].isel(z_t=k100)
    PON100 = ds["PON_FLUX_IN"].isel(z_t=k100)
    POP100 = ds["POP_FLUX_IN"].isel(z_t=k100)
    SiO2100 = ds["SiO2_FLUX_IN"].isel(z_t=k100)
    Piron100 = ds["P_iron_FLUX_IN"].isel(z_t=k100)

    # -------------------------------------------------------------------------
    # Nitrogen cycle
    # -------------------------------------------------------------------------

    Nfix = ds["diaz_Nfix"].where(ds["diaz_Nfix"] > 0)
    Nfix = Nfix.sum("z_t_150m", skipna=True)

    Denit = ds["DENITRIF"].sum("z_t", skipna=True)

    # -------------------------------------------------------------------------
    # Global totals
    # -------------------------------------------------------------------------

    NPP_tot.append(
        ((tarea * NPP100).sum(skipna=True).item())
        * (12.01 * 10 * 365.25 * 24 * 3600)
        * 1e-3
        * 1e-15
    )

    POC_tot.append(
        ((tarea * POC100).sum(skipna=True).item())
        * (12.01 * 10 * 365.25 * 24 * 3600)
        * 1e-6
        * 1e-15
    )

    PON_tot.append(
        ((tarea * PON100).sum(skipna=True).item())
        * (14.01 * 10 * 365.25 * 24 * 3600)
        * 1e-6
        * 1e-15
    )

    POP_tot.append(
        ((tarea * POP100).sum(skipna=True).item())
        * (10 * 365.25 * 24 * 3600)
        * 1e-6
        * 1e-12
    )

    Piron_tot.append(
        ((tarea * Piron100).sum(skipna=True).item())
        * (10 * 365.25 * 24 * 3600)
        * 1e-3
        * 1e-12
    )

    SiO2_tot.append(
        ((tarea * SiO2100).sum(skipna=True).item())
        * (10 * 365.25 * 24 * 3600)
        * 1e-6
        * 1e-12
    )

    Nfix_tot.append(
        ((tarea * Nfix).sum(skipna=True).item())
        * (14.01 * 10 * 365.25 * 24 * 3600)
        * 1e-3
        * 1e-12
    )

    Denit_tot.append(
        ((tarea * Denit).sum(skipna=True).item())
        * (14.01 * 10 * 365.25 * 24 * 3600)
        * 1e-3
        * 1e-12
    )

    spC_tot.append(
        ((tarea * spC.isel(z_t_150m=slice(0, 10))).sum(skipna=True).item())
        * (12.01 * 10)
        * 1e-3
        * 1e-12
    )

    diatC_tot.append(
        ((tarea * diatC.isel(z_t_150m=slice(0, 10))).sum(skipna=True).item())
        * (12.01 * 10)
        * 1e-3
        * 1e-12
    )

    diazC_tot.append(
        ((tarea * diazC.isel(z_t_150m=slice(0, 10))).sum(skipna=True).item())
        * (12.01 * 10)
        * 1e-3
        * 1e-12
    )

    biomass_tot.append(
        ((tarea * biomass.isel(z_t_150m=slice(0, 10))).sum(skipna=True).item())
        * (12.01 * 10)
        * 1e-3
        * 1e-12
    )

    ds.close()

# =============================================================================
# Convert to numpy arrays
# =============================================================================

NPP_tot = np.array(NPP_tot)
POC_tot = np.array(POC_tot)
PON_tot = np.array(PON_tot)
POP_tot = np.array(POP_tot)
Piron_tot = np.array(Piron_tot)
SiO2_tot = np.array(SiO2_tot)
Nfix_tot = np.array(Nfix_tot)
Denit_tot = np.array(Denit_tot)

spC_tot = np.array(spC_tot)
diatC_tot = np.array(diatC_tot)
diazC_tot = np.array(diazC_tot)
biomass_tot = np.array(biomass_tot)

# =============================================================================
# Stoichiometric ratios
# =============================================================================

CN_export = (POC_tot / 12.01) / (PON_tot / 14.01)
CP_export = (POC_tot / 12.01) / ((POP_tot * 1e-3))
NP_export = (PON_tot / 14.01) / ((POP_tot * 1e-3))

FeC_export = Piron_tot / (POC_tot / 12.01)
SiN_export = (SiO2_tot / 1e3) / (PON_tot / 14.01)

fracsp = spC_tot / biomass_tot
fracdiat = diatC_tot / biomass_tot
fracdiaz = diazC_tot / biomass_tot

/tmp/ipykernel_11344/282088201.py:7: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(fname)
/tmp/ipykernel_11344/282088201.py:7: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values